# Summaries Analysis

This notebook performs both **quantitative** and **qualitative** analyses of the summaries generated by the API, to build a complete picture of model summary performance.





##### 1. Quantitative Analysis
- Conduct a **statistical and exploratory data analysis** of the API results using Pandas and descriptive statistics.  
- Explore and refine the **visualizations** to better highlight differences and distributions.  
- Compare **model runtimes** (CPU vs GPU) for the same document to assess performance efficiency.  



##### 2. Qualitative Analysis
- **Garbage Detection:** Identify low-quality or nonsensical summaries (e.g., hallucinations or “garbage” outputs).  
  These can often be detected as **outliers** in the quantitative performance metrics.  
  DeepSeek has shown clear examples of this behavior.  
- **Summary Comparison:** Evaluate the **relative quality and performance** of summaries across different models, linking the findings with the quantitative metrics.


## Quantitative Analysis

### Load summaries

In [ ]:
# Libraries imports

import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
DATA_PATH = '/Users/sofi/Desktop/collectiveai/'
cuda_summaries_file = os.path.join(DATA_PATH, 'summarization-benchmark-results-cuda.json')
cpu_summaries_file = os.path.join(DATA_PATH, 'summarization-benchmark-results-cpu.json')

with open(cuda_summaries_file, 'r') as f:
    cuda_summaries = json.load(f)
with open(cpu_summaries_file, 'r') as f:
    cpu_summaries = json.load(f)


### Descriptive analysis

In [ ]:
summaries = cuda_summaries + cpu_summaries
df = pd.DataFrame(summaries)
df.head(4)

In [ ]:
import pprint
unique = df[['system_prompt','system_prompt_type']].drop_duplicates().reset_index(drop=True)

for _, row in unique.iterrows():
    print("Type:", row['system_prompt_type'])
    print("Prompt:")
    pprint.pprint(row['system_prompt'])
    print("---\n")

In [ ]:
df.info()

In [ ]:
def unique_vals(df, col):
    print(col)
    return print(f"Total {col}: {(df[col].nunique())} \n {(df[col].unique())} \n")


In [ ]:
for col in ['model','system_prompt','device','system_prompt_type','user_prompt','doc_path']:
    unique_vals(df,col)


#### Data engineer (ms -> mins)

In [ ]:
ms_cols = [c for c in df.columns if c.endswith('ms')]
ms_min_cols = [[c.replace('ms', 'min'), c] for c in ms_cols]
time_cols = [c[0] for c in ms_min_cols]

for cols in ms_min_cols:
    c, ms_col = cols
    df[c] = df[ms_col] / 1000 / 60

df.head(2)


time_cols

#### Means ( minutes) 

The mean imput tokens by model are:

In [ ]:
means_by_model = df.groupby('model').mean().sort_values('tokens_per_second')#[['tokens_per_second','model_duration_ms','total_tokens','input_duration_ms','output_duration_ms']]
means_by_model

In [ ]:
for filter, df_filtered in [("all", df), ("cuda", df[df['device']=='cuda']), ("cpu", df[df['device']=='cpu'])]:
    fig, axes = plt.subplots(ncols=2, figsize=(10, 5))
    label = filter if filter != "all" else filter + " devices"

    sns.barplot(
        data=df_filtered,
        x="model",
        y="tokens_per_second",
        ax=axes[0],
        order=vc.index if 'vc' in globals() else None, label = label
    )
    axes[0].set_title("Generation Speed by Model")
    axes[0].set_ylabel("Tokens per Second")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].grid(axis="y", linestyle="--", alpha=0.7)

    sns.barplot(
        data=df_filtered,
        x="model",
        y="measured_duration_min",
        ax=axes[1],
        order=vc.index if 'vc' in globals() else None, label = label
    )
    axes[1].set_title("Total Processing Time by Model")
    axes[1].set_ylabel("Duration (min)")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].grid(axis="y", linestyle="--", alpha=0.7)

    plt.tight_layout()
    plt.show()


### Gap Analysis (CPU - CUDA)

#### Time Gap

In [ ]:
# time gap between CPU and CUDA (cpu_ms - cuda_ms)
pivot = df.groupby(['model', 'device'])['measured_duration_min'].mean().unstack()
pivot.head()

In [ ]:
# time gap between CPU and CUDA (cpu_ms - cuda_ms)
pivot = df.groupby(['model', 'device'])['measured_duration_min'].mean().unstack()
if {'cpu', 'cuda'}.issubset(pivot.columns):
    # ensure no missing values and work on a copy
    pivot = pivot.reindex(pivot.index).fillna(0).copy()

    # reset index so 'model' becomes a column
    pivot = pivot.reset_index().rename(columns={'index': 'model'})

    # ensure numeric types and compute difference (values are already in minutes)
    pivot['cpu'] = pd.to_numeric(pivot['cpu'], errors='coerce').fillna(0)
    pivot['cuda'] = pd.to_numeric(pivot['cuda'], errors='coerce').fillna(0)
    pivot['cpu_minus_cuda_min'] = pivot['cpu'] - pivot['cuda']

    # build a 1-D list of model names ordered by the gap
    order = list(pivot.sort_values('cpu_minus_cuda_min', ascending=False)['model'])

    plt.figure(figsize=(12, 5))
    sns.barplot(
        data=pivot,
        x='model',
        y='cpu_minus_cuda_min',
        order=order
    )
    plt.axhline(0, color='k', linestyle='--', linewidth=0.8)
    plt.xticks(rotation=45)
    plt.ylabel('CPU - CUDA duration (min)')
    plt.title('Time gap between CPU and CUDA by model (CPU - CUDA)')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data to compute CPU vs CUDA gap (missing 'cpu' or 'cuda' device rows).")

#### Token per second gap

In [ ]:
# time gap between CPU and CUDA (cpu_ms - cuda_ms)
pivot = df.groupby(['model', 'device'])['tokens_per_second'].mean().unstack()
if {'cpu', 'cuda'}.issubset(pivot.columns):
    # ensure no missing values and work on a copy
    pivot = pivot.reindex(pivot.index).fillna(0).copy()

    # reset index so 'model' becomes a column
    pivot = pivot.reset_index().rename(columns={'index': 'model'})

    # ensure numeric types and compute difference (values are already in minutes)
    pivot['cpu'] = pd.to_numeric(pivot['cpu'], errors='coerce').fillna(0)
    pivot['cuda'] = pd.to_numeric(pivot['cuda'], errors='coerce').fillna(0)
    pivot['cpu_minus_cuda_min'] = pivot['cpu'] - pivot['cuda']

    # build a 1-D list of model names ordered by the gap
    order = list(pivot.sort_values('cpu_minus_cuda_min', ascending=False)['model'])

    plt.figure(figsize=(12, 5))
    sns.barplot(
        data=pivot,
        x='model',
        y='cpu_minus_cuda_min',
        order=order
    )
    plt.axhline(0, color='k', linestyle='--', linewidth=0.8)
    plt.xticks(rotation=45)
    plt.ylabel('CPU - CUDA tokens per second gap')
    plt.title('Token per second gap between CPU and CUDA by model (CPU - CUDA)')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data to compute CPU vs CUDA gap (missing 'cpu' or 'cuda' device rows).")

In [ ]:
df.groupby(['model', 'device'])['tokens_per_second'].mean().unstack()

In [ ]:

pivot_token = df.groupby(['model', 'device'])['tokens_per_second'].mean().unstack()
pivot_dur = df.groupby(['model', 'device'])['measured_duration_min'].mean().unstack()

# prepare a list of models to keep consistent ordering (try to reuse vc order if present)
models = list(vc.index) if 'vc' in globals() else sorted(df['model'].unique())

# build a safe summary even if one device is missing
def safe_gap(pivot, models):
    # ensure index contains all models and missing values filled with 0
    p = pivot.reindex(models).fillna(0).copy().reset_index().rename(columns={'index': 'model'})
    # coerce numeric and add cpu/cuda columns if missing
    p['cpu'] = pd.to_numeric(p.get('cpu', 0), errors='coerce').fillna(0)
    p['cuda'] = pd.to_numeric(p.get('cuda', 0), errors='coerce').fillna(0)
    return p

p_token = safe_gap(pivot_token, models)
p_dur = safe_gap(pivot_dur, models)

model_summary = pd.DataFrame({
    'model': p_token['model'],
    'gap_input_tokens': p_token['cpu'] - p_token['cuda'],
    # convert minutes gap to seconds
    'gap_total_duration_min': (p_dur['cpu'] - p_dur['cuda']),
})

# bubble size: proportional to number of samples per model (scaled for plotting)
counts = df.groupby('model').size().reindex(model_summary['model']).fillna(1)
model_summary['bubble_size'] = (counts / counts.max()) * 400 + 50

# ensure columns have expected dtypes
model_summary['gap_input_tokens'] = pd.to_numeric(model_summary['gap_input_tokens'], errors='coerce').fillna(0)
model_summary['gap_total_duration_min'] = pd.to_numeric(model_summary['gap_total_duration_min'], errors='coerce').fillna(0)

model_summary.head()


In [ ]:

pivot_token = df.groupby(['model', 'device'])['tokens_per_second'].mean().unstack()
pivot_dur = df.groupby(['model', 'device'])['measured_duration_min'].mean().unstack()

# prepare a list of models to keep consistent ordering (try to reuse vc order if present)
models = list(vc.index) if 'vc' in globals() else sorted(df['model'].unique())

# build a safe summary even if one device is missing
def safe_gap(pivot, models):
    # ensure index contains all models and missing values filled with 0
    p = pivot.reindex(models).fillna(0).copy().reset_index().rename(columns={'index': 'model'})
    # coerce numeric and add cpu/cuda columns if missing
    p['cpu'] = pd.to_numeric(p.get('cpu', 0), errors='coerce').fillna(0)
    p['cuda'] = pd.to_numeric(p.get('cuda', 0), errors='coerce').fillna(0)
    return p

p_token = safe_gap(pivot_token, models)
p_dur = safe_gap(pivot_dur, models)

model_summary = pd.DataFrame({
    'model': p_token['model'],
    'gap_tokens': p_token['cpu'] - p_token['cuda'],
    # convert minutes gap to seconds
    'gap_total_duration_min': (p_dur['cpu'] - p_dur['cuda']),
})

# bubble size: proportional to number of samples per model (scaled for plotting)
counts = df.groupby('model').size().reindex(model_summary['model']).fillna(1)
model_summary['bubble_size'] = (counts / counts.max()) * 400 + 50

# ensure columns have expected dtypes
model_summary['gap_tokens'] = pd.to_numeric(model_summary['gap_tokens'], errors='coerce').fillna(0)
model_summary['gap_total_duration_min'] = pd.to_numeric(model_summary['gap_total_duration_min'], errors='coerce').fillna(0)

model_summary.head()


fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(
    model_summary["gap_tokens"],
    model_summary["gap_total_duration_min"],
    #s=model_summary["bubble_size"],
    alpha=0.7,
    color="#1f77b4",
    edgecolors="black",
)
for _, row in model_summary.iterrows():
    ax.text(
        row["gap_tokens"],
        row["gap_total_duration_min"],
        row["model"],
        fontsize=9,
        ha="left",
        va="bottom",
        alpha=0.9,
    )
ax.set_ylabel("Average Total Duration Gap (min)")
ax.set_xlabel("Average Tokens per second")
#ax.set_ylabel("Average Total Duration (s)")
ax.grid(True, linestyle="--", alpha=0.6)
ax.set_title("CPU vs CUDA Performance Gap by Model")
plt.show()

In [ ]:
# Attempt to identify parameter scale (in billions) from model names
def get_model_parameter_scale(model_name):
    import re
    if not model_name:
        return None

    known_params = {
        "gemma3:12b": 12,
        "gemma3:40b": 40,
        "gemma3:20b": 20,
        "gemma3:22b": 22,
        "gemma3:38b": 38,
        "llama3.1:8b": 8,
        "llama3.2:3b": 3,
        "llama3.2:11b": 11,
        "deepseek-r1:1.1b": 1.1,
        "deepseek-r1:1b": 1,
        "phi-3:3.8b": 3.8,
        "phi-4:14b": 14,
        "qwen3:3.8b": 3.8,
        "qwen3:14b": 14,
        "qwen3:72b": 72,
    }

    model_name_lower = model_name.lower()
    if model_name_lower in known_params:
        return known_params[model_name_lower]

    match = re.search(r"(\d+(?:\.\d+)?)\s*b", model_name_lower)
    if match:
        try:
            return float(match.group(1))
        except ValueError:
            return None
    return None

In [ ]:

pivot_token = df.groupby(['model', 'device'])['tokens_per_second'].mean().unstack()
pivot_dur = df.groupby(['model', 'device'])['measured_duration_min'].mean().unstack()

# prepare a list of models to keep consistent ordering (try to reuse vc order if present)
models = list(vc.index) if 'vc' in globals() else sorted(df['model'].unique())

# build a safe summary even if one device is missing
def safe_gap(pivot, models):
    # ensure index contains all models and missing values filled with 0
    p = pivot.reindex(models).fillna(0).copy().reset_index().rename(columns={'index': 'model'})
    # coerce numeric and add cpu/cuda columns if missing
    p['cpu'] = pd.to_numeric(p.get('cpu', 0), errors='coerce').fillna(0)
    p['cuda'] = pd.to_numeric(p.get('cuda', 0), errors='coerce').fillna(0)
    return p

p_token = safe_gap(pivot_token, models)
p_dur = safe_gap(pivot_dur, models)

model_summary = pd.DataFrame({
    'model': p_token['model'],
    'gap_tokens': p_token['cuda'] - p_token['cpu'],
    # convert minutes gap to seconds
    'gap_total_duration_min': (p_dur['cuda'] - p_dur['cpu']),
})

# bubble size: proportional to number of samples per model (scaled for plotting)
#counts = df.groupby('model').size().reindex(model_summary['model']).fillna(1)
#model_summary['bubble_size'] = (counts / counts.max()) * 400 + 50
model_summary['param_scale'] = model_summary['model'].apply(get_model_parameter_scale)
# ensure columns have expected dtypes
model_summary['gap_tokens'] = pd.to_numeric(model_summary['gap_tokens'], errors='coerce').fillna(0)
model_summary['gap_total_duration_min'] = pd.to_numeric(model_summary['gap_total_duration_min'], errors='coerce').fillna(0)

model_summary.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# generate a unique color per model
palette = sns.color_palette( n_colors=len(model_summary))
model_summary.sort_values("param_scale", inplace=True)
model_colors = dict(zip(model_summary["model"], palette))

fig, ax = plt.subplots(figsize=(12, 6))
i = 1
# plot bubbles with model-specific colors
for _, row in model_summary.iterrows():
    color = model_colors[row["model"]]
    ax.scatter(
        
        row["param_scale"], #row["gap_total_duration_min"],
        row["gap_tokens"],
        s=row["param_scale"] * 100,
        alpha=0.7,
        color=color,
        edgecolors="black",
        linewidth=0.7,
    )
    ax.text(
        row["param_scale"] + (0.5*(-1)**i), #row["gap_total_duration_min"]
        row["gap_tokens"],
        row["model"],
        fontsize=9,
        ha="left",
        va="bottom",
        alpha=1,
        fontweight='bold',
        color=color,
    )
    i += 1
    
    # add a label for the bubble size

ax.set_ylabel("Average Tokens per Second")
ax.set_xlabel("Parameter Scale (Billion Parameters)")
ax.grid(True, linestyle="--", alpha=0.6)
ax.set_title("CUDA - CPU Gap Token per Second by Model Parameter Scale per Model")

# optional legend showing color per model
handles = [
    plt.Line2D([0], [0], marker="o", color="w", label=model,
               markerfacecolor=color, markersize=8, markeredgecolor="black")
    for model, color in model_colors.items()
]
ax.legend(handles=handles, title="Model", loc="best")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# generate a unique color per model
palette = sns.color_palette( n_colors=len(model_summary))
model_summary.sort_values("param_scale", inplace=True)
model_colors = dict(zip(model_summary["model"], palette))

fig, ax = plt.subplots(figsize=(12, 6))
i = 1
# plot bubbles with model-specific colors
for _, row in model_summary.iterrows():
    color = model_colors[row["model"]]
    ax.scatter(
        
        row["param_scale"], #row["gap_total_duration_min"],
        row["gap_total_duration_min"],
        s=row["param_scale"] * 100,
        alpha=0.7,
        color=color,
        edgecolors="black",
        linewidth=0.7,
    )
    ax.text(
        row["param_scale"] + (0.5*(-1)**i), #row["gap_total_duration_min"]
        row["gap_total_duration_min"],
        row["model"],
        fontsize=9,
        ha="left",
        va="bottom",
        alpha=1,
        fontweight='bold',
        color=color,
    )
    i += 1
    
    # add a label for the bubble size

ax.set_ylabel("Average Total Duration Gap (min)")
ax.set_xlabel("Parameter Scale (Billion Parameters)")
ax.grid(True, linestyle="--", alpha=0.6)
ax.set_title("CUDA - CPU Gap Token per Second by Model Parameter Scale per Model")

# optional legend showing color per model
handles = [
    plt.Line2D([0], [0], marker="o", color="w", label=model,
               markerfacecolor=color, markersize=8, markeredgecolor="black")
    for model, color in model_colors.items()
]
ax.legend(handles=handles, title="Model", loc="best")

plt.tight_layout()
plt.show()


In [ ]:
(df.iloc[0].output_tokens / df.iloc[0].output_duration_ms)*1000 # it is tokens_per_second

In [ ]:
from aymurai.utils.json_data import load_json, save_json

save_json(summaries,DATA_PATH + "summaries.json", )

In [ ]:
df.iloc[0].input_duration_ms + df.iloc[0].output_duration_ms   # total duration in ms

## Qualitative Analysis

### Garbage Detection - Outliers 

In [ ]:
df.describe()

max output_tokens is 163840, same value as max input token, so, it is truncated. We will see examples

In [ ]:
df_truncated = df[df['output_tokens']>16000]
print('Truncated summary analysis: \ndoc paths: \n',df_truncated.doc_path.unique())
print('Models: \n',df_truncated.model.unique())
df_truncated.head()

In [ ]:
df_truncated.groupby(['model','doc_path','system_prompt_type']).count()

### Output outliers visualizations

In [ ]:
for idx, txt in enumerate(df_truncated['chat_response'].astype(str).fillna('')):
    print(f"+++++++++++++ Truncated summary {idx} (model={df_truncated.iloc[idx].get('model', '')}, system_prompt_type={df_truncated.iloc[idx].get('system_prompt_type', '')}, doc_path={df_truncated.iloc[idx].get('doc_path', '')}) +++++++++++++")
    print(txt[-100:])
    print("\n")

In [ ]:
# summaries -> df.chat_response.nunique()
# if input_token = 16384, it is trunckated
# model_duration_ms  [miliseconds] -> total time to analyze 

In [ ]:
set(df.system_prompt)

In [ ]:
set(df.system_prompt_type)

In [ ]:
cuda_summaries[1]